Here is the official webiste to teach how to use weaviate. Before running this notebook. You need to turn on the weaviate docker

In [1]:
import weaviate
from weaviate.classes.config import Configure
import requests, json

c:\Users\jason\miniconda3\envs\ai\Lib\site-packages\authlib\_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey


In [2]:
client = weaviate.connect_to_local(host='127.0.0.1', port=8080)
print("Ready:", client.is_ready())  # Should print: True
print("Meta:", client.get_meta())   # Shows version, modules, etc.

Ready: True
Meta: {'grpcMaxMessageSize': 104858000, 'hostname': 'http://[::]:8080', 'modules': {'generative-anthropic': {'documentationHref': 'https://docs.anthropic.com/en/api/getting-started', 'name': 'Generative Search - Anthropic'}, 'generative-anyscale': {'documentationHref': 'https://docs.anyscale.com/endpoints/overview', 'name': 'Generative Search - Anyscale'}, 'generative-aws': {'documentationHref': 'https://docs.aws.amazon.com/bedrock/latest/APIReference/welcome.html', 'name': 'Generative Search - AWS'}, 'generative-cohere': {'documentationHref': 'https://docs.cohere.com/reference/chat', 'name': 'Generative Search - Cohere'}, 'generative-databricks': {'documentationHref': 'https://docs.databricks.com/en/machine-learning/foundation-models/api-reference.html#completion-task', 'name': 'Generative Search - Databricks'}, 'generative-friendliai': {'documentationHref': 'https://docs.friendli.ai/openapi/create-chat-completions', 'name': 'Generative Search - FriendliAI'}, 'generative-g

In [3]:
if not client.collections.exists("TrialConnection"):
    questions = client.collections.create(
        name="TrialConnection",
        vector_config=Configure.Vectors.text2vec_transformers()
    )
else:
    questions = client.collections.get("TrialConnection")

### loading data on website

In [4]:
resp = requests.get(
    "https://raw.githubusercontent.com/weaviate-tutorials/quickstart/main/data/jeopardy_tiny.json"
)
data = json.loads(resp.text)

In [5]:
print(type(data))
print('length of data: ',len(data))
data

<class 'list'>
length of data:  10


[{'Category': 'SCIENCE',
  'Question': 'This organ removes excess glucose from the blood & stores it as glycogen',
  'Answer': 'Liver'},
 {'Category': 'ANIMALS',
  'Question': "It's the only living mammal in the order Proboseidea",
  'Answer': 'Elephant'},
 {'Category': 'ANIMALS',
  'Question': 'The gavial looks very much like a crocodile except for this bodily feature',
  'Answer': 'the nose or snout'},
 {'Category': 'ANIMALS',
  'Question': 'Weighing around a ton, the eland is the largest species of this animal in Africa',
  'Answer': 'Antelope'},
 {'Category': 'ANIMALS',
  'Question': 'Heaviest of all poisonous snakes is this North American rattlesnake',
  'Answer': 'the diamondback rattler'},
 {'Category': 'SCIENCE',
  'Question': "2000 news: the Gunnison sage grouse isn't just another northern sage grouse, but a new one of this classification",
  'Answer': 'species'},
 {'Category': 'SCIENCE',
  'Question': 'A metal that is ductile can be pulled into this while cold & under pressur

### add data object

In [6]:
with questions.batch.fixed_size(batch_size=200) as batch:
    for d in data:
        batch.add_object(
            {
                "answer": d["Answer"],
                "question": d["Question"],
                "category": d["Category"],
            }
        )
        if batch.number_errors > 10:
            print("Batch import stopped due to excessive errors.")
            break

failed_objects = questions.batch.failed_objects
if failed_objects:
    print(f"Number of failed imports: {len(failed_objects)}")
    print(f"First failed object: {failed_objects[0]}")

### query

In [7]:
response = questions.query.near_text(
    query="biology",
    limit=2
)

In [8]:
for obj in response.objects:
    print(json.dumps(obj.properties, indent=2))

{
  "category": "SCIENCE",
  "question": "In 1953 Watson & Crick built a model of the molecular structure of this, the gene-carrying substance",
  "answer": "DNA"
}
{
  "category": "ANIMALS",
  "question": "It's the only living mammal in the order Proboseidea",
  "answer": "Elephant"
}


In [7]:
client.close()